#  EAS-PQC Framework: NIST Algorithms Only
## Post-Quantum Cryptography Evaluation for IoT Devices

---

### 🎯 Algorithm Selection Criteria

We evaluate **NIST-standardized Post-Quantum Cryptography algorithms (FIPS 203-205)** and **selected Round 4 finalists**. This selection represents the algorithms most likely to be deployed in real-world IoT systems, ensuring our findings are directly applicable to industry implementations.

**Our evaluation includes:**
- **ML-KEM variants (FIPS 203):** All parameter sets (512, 768, 1024)
- **ML-DSA variants (FIPS 204):** All parameter sets (44, 65, 87)
- **SLH-DSA variants (FIPS 205):** Representative SPHINCS+ parameter sets
- **BIKE:** Round 4 code-based KEM candidate
- **HQC:** Round 4 code-based KEM candidate
- **Falcon (FN-DSA):** Round 4 lattice signature candidate

While the PQM4 benchmark suite contains **92 cryptographic schemes**, we focus on the **25 NIST-designated algorithms** to maintain relevance to standards-compliant IoT deployments.

## 📦 Part 1: Setup

In [16]:
!pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 📁 Part 2: Upload CSV

In [17]:
from google.colab import files

print("📤 Upload pqc_iot_classification.csv")
uploaded = files.upload()
print("\n✅ File uploaded!")

📤 Upload pqc_iot_classification.csv


Saving pqc_iot_classification.csv to pqc_iot_classification (1).csv

✅ File uploaded!


## 🏗️ Part 3: Define Device Classes (RFC 7228)

In [18]:
device_classes = {
    'Class 0': {'ram': 4096, 'flash': 32768, 'desc': 'Sensors (4KB RAM)'},
    'Class 1': {'ram': 10240, 'flash': 102400, 'desc': 'Smart Sensors (10KB RAM)'},
    'Class 2': {'ram': 51200, 'flash': 256000, 'desc': 'Gateways (50KB RAM)'}
}

print("🏗️ Device Classes (RFC 7228):")
for name, info in device_classes.items():
    print(f"   {name}: {info['ram']//1024} KB RAM, {info['flash']//1024} KB Flash")

🏗️ Device Classes (RFC 7228):
   Class 0: 4 KB RAM, 32 KB Flash
   Class 1: 10 KB RAM, 100 KB Flash
   Class 2: 50 KB RAM, 250 KB Flash


## 🔍 Part 4: NIST Algorithm Filter (CRITICAL - This is where we filter!)

In [19]:
def is_nist_algorithm(scheme_name):
    """
    Check if an algorithm is a NIST PQC standardized or Round 4 candidate.

    Returns: True if NIST algorithm, False otherwise
    """
    scheme_lower = scheme_name.lower()

    # NIST Standardized (FIPS 203-205)
    nist_standardized = [
        'ml-kem',      # FIPS 203
        'ml-dsa',      # FIPS 204
        'sphincs'      # FIPS 205 (SLH-DSA)
    ]

    # NIST Round 4 Finalists
    nist_round4 = [
        'bike',        # Code-based KEM
        'hqc',         # Code-based KEM
        'fndsa',       # Falcon signature
        'falcon'       # Alternative name for Falcon
    ]

    # Check if algorithm matches any NIST pattern
    all_nist = nist_standardized + nist_round4

    return any(nist_name in scheme_lower for nist_name in all_nist)


def categorize_nist_algorithm(scheme_name):
    """Categorize NIST algorithm by family."""
    scheme_lower = scheme_name.lower()

    if 'ml-kem' in scheme_lower:
        return 'ML-KEM (Lattice KEM)'
    elif 'ml-dsa' in scheme_lower:
        return 'ML-DSA (Lattice Signature)'
    elif 'sphincs' in scheme_lower:
        return 'SLH-DSA/SPHINCS+ (Hash Signature)'
    elif 'bike' in scheme_lower:
        return 'BIKE (Code-based KEM)'
    elif 'hqc' in scheme_lower:
        return 'HQC (Code-based KEM)'
    elif 'fndsa' in scheme_lower or 'falcon' in scheme_lower:
        return 'Falcon/FN-DSA (Lattice Signature)'
    else:
        return 'Other NIST'

print("✅ NIST filter functions ready")

✅ NIST filter functions ready


## 📊 Part 5: Load Data and Filter for NIST Algorithms Only

In [20]:
# Load CSV
df_all = pd.read_csv('pqc_iot_classification.csv')

print(f"📊 Full Dataset:")
print(f"   Total rows: {len(df_all)}")
print(f"   Unique schemes: {df_all['Scheme'].nunique()}")

# CRITICAL FIX: Remove duplicate percentage rows
# CSV has 2 rows per scheme: bytes (correct) and percentages (wrong)
print(f"\n🔧 Filtering duplicate rows...")
df_all = df_all[df_all['Decapsulation [bytes]'] > 100].copy()
print(f"   Rows after filter: {len(df_all)}")
print(f"   Unique schemes after filter: {df_all['Scheme'].nunique()}")

# Filter for NIST algorithms ONLY
df_all['is_nist'] = df_all['Scheme'].apply(is_nist_algorithm)
df = df_all[df_all['is_nist']].copy()

print(f"\n🔍 After NIST Filtering:")
print(f"   NIST algorithms: {len(df)} rows")
print(f"   NIST unique schemes: {df['Scheme'].nunique()}")
print(f"   Non-NIST excluded: {len(df_all) - len(df)} rows")

# Add category
df['NIST_Family'] = df['Scheme'].apply(categorize_nist_algorithm)

print(f"\n📋 NIST Algorithms by Family:")
family_counts = df.groupby('NIST_Family')['Scheme'].nunique()
for family, count in family_counts.items():
    print(f"   {family:<40} → {count} unique algorithms")

print("\n✅ Dataset filtered to NIST algorithms only!")

📊 Full Dataset:
   Total rows: 276
   Unique schemes: 92

🔧 Filtering duplicate rows...
   Rows after filter: 138
   Unique schemes after filter: 92

🔍 After NIST Filtering:
   NIST algorithms: 41 rows
   NIST unique schemes: 25
   Non-NIST excluded: 97 rows

📋 NIST Algorithms by Family:
   BIKE (Code-based KEM)                    → 2 unique algorithms
   Falcon/FN-DSA (Lattice Signature)        → 2 unique algorithms
   HQC (Code-based KEM)                     → 3 unique algorithms
   ML-DSA (Lattice Signature)               → 3 unique algorithms
   ML-KEM (Lattice KEM)                     → 3 unique algorithms
   SLH-DSA/SPHINCS+ (Hash Signature)        → 12 unique algorithms

✅ Dataset filtered to NIST algorithms only!


## Part 6: Extract Best Implementation per NIST Algorithm

In [22]:
def get_nist_algorithms(df, prefer_impl='clean'):
    """
    Extract one row per NIST algorithm (best implementation).
    Prefers 'clean' implementation, falls back to others.
    """

    # Security level corrections (CSV has wrong values for ML-DSA)
    SECURITY_CORRECT = {
        'ml-dsa-44': 2,
        'ml-dsa-65': 3,
        'ml-dsa-87': 5,
        'ml-kem-512': 1,
        'ml-kem-768': 3,
        'ml-kem-1024': 5,
        'sphincs-sha2-128f-simple': 1,
        'sphincs-sha2-128s-simple': 1,
        'sphincs-sha2-192f-simple': 3,
        'sphincs-sha2-192s-simple': 3,
        'sphincs-sha2-256f-simple': 5,
        'sphincs-sha2-256s-simple': 5,
        'hqc-128': 1,
        'hqc-192': 3,
        'hqc-256': 5,
        'bikel1': 1,
        'bikel3': 3,
        'fndsa_provisional-512': 1,
        'fndsa_provisional-1024': 5,
    }

    result = []

    for scheme in df['Scheme'].unique():
        scheme_data = df[df['Scheme'] == scheme]

        # Prefer 'clean' implementation
        clean_match = scheme_data[scheme_data['Implementation'] == prefer_impl]

        # THIS IF/ELSE MUST BE INDENTED INSIDE THE FOR LOOP
        if len(clean_match) > 0:
            row = clean_match.iloc[0]
        else:
            # Fallback order: prioritize stack-optimized, then m4f
            # This ensures BIKE uses m4f (not opt which has different trade-offs)
            fallback_order = ['m4fstack', 'm4f', 'm4fspeed', 'opt']
            row = None
            for impl in fallback_order:
                fallback = scheme_data[scheme_data['Implementation'] == impl]
                if len(fallback) > 0:
                    row = fallback.iloc[0]
                    print(f"   Using {impl} for {scheme} (no clean available)")
                    break

            if row is None:
                row = scheme_data.iloc[0]  # Ultimate fallback

        # Determine NIST family
        scheme_lower = scheme.lower()
        if 'ml-kem' in scheme_lower or 'kyber' in scheme_lower:
            family = 'ML-KEM (Lattice KEM)'
        elif 'ml-dsa' in scheme_lower or 'dilithium' in scheme_lower:
            family = 'ML-DSA (Lattice Signature)'
        elif 'sphincs' in scheme_lower or 'slh-dsa' in scheme_lower:
            family = 'SLH-DSA/SPHINCS+ (Hash Signature)'
        elif 'bike' in scheme_lower:
            family = 'BIKE (Code-based KEM)'
        elif 'hqc' in scheme_lower:
            family = 'HQC (Code-based KEM)'
        elif 'falcon' in scheme_lower or 'fndsa' in scheme_lower:
            family = 'Falcon/FN-DSA (Lattice Signature)'
        else:
            family = 'Other'

        # Calculate EAS scores for each class
        eas_c0 = calculate_eas_pqc(
            int(row['Total_Cycles']),
            int(row['Max_RAM']),
            int(row['Flash']),
            SECURITY_CORRECT.get(scheme.lower(), int(row['Security_Level'])),
            device_classes['Class 0']['ram'],
            device_classes['Class 0']['flash']
        )

        eas_c1 = calculate_eas_pqc(
            int(row['Total_Cycles']),
            int(row['Max_RAM']),
            int(row['Flash']),
            SECURITY_CORRECT.get(scheme.lower(), int(row['Security_Level'])),
            device_classes['Class 1']['ram'],
            device_classes['Class 1']['flash']
        )

        eas_c2 = calculate_eas_pqc(
            int(row['Total_Cycles']),
            int(row['Max_RAM']),
            int(row['Flash']),
            SECURITY_CORRECT.get(scheme.lower(), int(row['Security_Level'])),
            device_classes['Class 2']['ram'],
            device_classes['Class 2']['flash']
        )

        result.append({
            'Scheme': scheme,
            'NIST_Family': family,
            'Implementation': row['Implementation'],
            'Total_Cycles': int(row['Total_Cycles']),
            'Max_RAM': int(row['Max_RAM']),
            'Flash': int(row['Flash']),
            'Security_Level': SECURITY_CORRECT.get(scheme.lower(), int(row['Security_Level'])),
            'EAS_Class_0': eas_c0,
            'EAS_Class_1': eas_c1,
            'EAS_Class_2': eas_c2
        })

    return pd.DataFrame(result)


# Extract best implementation per algorithm
nist_algos = get_nist_algorithms(df, prefer_impl='clean')

print(f"✅ Extracted {len(nist_algos)} NIST algorithms (one per scheme)")
print(f"\n📋 Complete List:")
for idx, row in nist_algos.iterrows():
    impl_note = '' if row['Implementation'] == 'clean' else f" ({row['Implementation']})"
    print(f"   {row['Scheme']:<40} {row['NIST_Family']:<40}{impl_note}")

   Using m4f for bikel1 (no clean available)
   Using m4f for bikel3 (no clean available)
   Using m4f for fndsa_provisional-1024 (no clean available)
   Using m4f for fndsa_provisional-512 (no clean available)
✅ Extracted 25 NIST algorithms (one per scheme)

📋 Complete List:
   bikel1                                   BIKE (Code-based KEM)                    (m4f)
   bikel3                                   BIKE (Code-based KEM)                    (m4f)
   hqc-128                                  HQC (Code-based KEM)                    
   hqc-192                                  HQC (Code-based KEM)                    
   hqc-256                                  HQC (Code-based KEM)                    
   ml-kem-1024                              ML-KEM (Lattice KEM)                    
   ml-kem-512                               ML-KEM (Lattice KEM)                    
   ml-kem-768                               ML-KEM (Lattice KEM)                    
   fndsa_provisional-1024      

## 🧮 Part 7: Calculate EAS Scores

In [23]:
def calculate_eas_pqc(cycles, ram, flash, security_level, ram_max, flash_max):
    """Calculate EAS-PQC score (Equation 1 from paper)."""
    if ram > ram_max or flash > flash_max:
        return 0.0

    ws_map = {1: 1.0, 2: 1.0, 3: 1.5, 4: 1.5, 5: 2.0}
    ws = ws_map.get(security_level, 1.0)
    eff = 1.0 / np.log10(cycles)  # Paper Equation 1: Eff = 1/log10(Ca)
    adapt = (1.0 - ram/ram_max) + (1.0 - flash/flash_max)
    eas = (ws * 0.4) + (eff * 0.3) + (adapt * 0.3)
    return round(eas, 3)

print("🔄 Calculating EAS scores...\n")

for class_name, class_info in device_classes.items():
    col_name = f"EAS_{class_name.replace(' ', '_')}"

    nist_algos[col_name] = nist_algos.apply(
        lambda row: calculate_eas_pqc(
            row['Total_Cycles'],
            row['Max_RAM'],
            row['Flash'],
            row['Security_Level'],
            class_info['ram'],
            class_info['flash']
        ),
        axis=1
    )

    feasible = (nist_algos[col_name] > 0).sum()
    print(f"   {class_name}: {feasible}/{len(nist_algos)} feasible")

print("\n✅ All EAS scores calculated!")

🔄 Calculating EAS scores...

   Class 0: 0/25 feasible
   Class 1: 13/25 feasible
   Class 2: 15/25 feasible

✅ All EAS scores calculated!


## 📊 Part 8: Generate Table II for Paper

### Strategy: Select representative algorithms from each NIST family

In [24]:
print("\n" + "="*80)
print("📊 TABLE II: Raw Performance Benchmarks (Representative NIST Algorithms)")
print("="*80)

# Strategy: Select best 2-3 from each family for paper
table_ii_selection = {
    'ML-KEM (Lattice KEM)': 3,              # All 3 variants
    'ML-DSA (Lattice Signature)': 3,        # All 3 variants
    'SLH-DSA/SPHINCS+ (Hash Signature)': 3, # 3 representative variants
    'HQC (Code-based KEM)': 2,              # 2 variants
    'BIKE (Code-based KEM)': 2,             # Both variants
    'Falcon/FN-DSA (Lattice Signature)': 2  # Both variants
}

table_ii_data = []

for family, num_to_select in table_ii_selection.items():
    family_algos = nist_algos[nist_algos['NIST_Family'] == family]

    # Sort by EAS score (Class 2) and take top N
    top_n = family_algos.nlargest(num_to_select, 'EAS_Class_2')

    for idx, row in top_n.iterrows():
        table_ii_data.append({
            'Scheme': row['Scheme'].upper(),
            'Family': family,
            'Total Cycles': f"{int(row['Total_Cycles']):,}",
            'RAM (Bytes)': f"{int(row['Max_RAM']):,}",
            'Flash (Bytes)': f"{int(row['Flash']):,}",
            'Security Level': int(row['Security_Level'])
        })

table_ii_df = pd.DataFrame(table_ii_data)

print(f"\n✅ Selected {len(table_ii_df)} representative NIST algorithms for Table II:\n")
display(table_ii_df)

print("\n💡 Selection criteria: Top algorithms per family by EAS score")


📊 TABLE II: Raw Performance Benchmarks (Representative NIST Algorithms)

✅ Selected 15 representative NIST algorithms for Table II:



,Scheme,Family,Total Cycles,RAM (Bytes),Flash (Bytes),Security Level
0,ML-KEM-1024,ML-KEM (Lattice KEM),"5,264,741","20,352","6,160",5
1,ML-KEM-768,ML-KEM (Lattice KEM),"3,514,931","14,480","5,120",3
2,ML-KEM-512,ML-KEM (Lattice KEM),"2,185,051","9,560","5,116",1
3,ML-DSA-44,ML-DSA (Lattice Signature),"11,863,456","51,976","8,212",2
4,ML-DSA-65,ML-DSA (Lattice Signature),"18,941,894","79,624","7,724",3
5,ML-DSA-87,ML-DSA (Lattice Signature),"26,531,579","122,740","8,036",5
6,SPHINCS-SHAKE-256F-SIMPLE,SLH-DSA/SPHINCS+ (Hash Signature),"4,335,038,729","7,928","4,720",5
7,SPHINCS-SHA2-256F-SIMPLE,SLH-DSA/SPHINCS+ (Hash Signature),"1,477,654,775","8,460","5,728",5
8,SPHINCS-SHAKE-256S-SIMPLE,SLH-DSA/SPHINCS+ (Hash Signature),"41,430,508,488","8,220","5,076",5
9,HQC-128,HQC (Code-based KEM),"317,925,277","55,892","18,628",1



💡 Selection criteria: Top algorithms per family by EAS score


## 📊 Part 9: Generate Table III (EAS Scores)

In [ ]:
print("\n" + "="*80)
print("📊 TABLE III: EAS-PQC Scores by Device Class")
print("="*80)

# Use same algorithms as Table II
table_iii_data = []

for idx, row_ii in table_ii_df.iterrows():
    scheme = row_ii['Scheme'].lower()

    # Find in nist_algos
    match = nist_algos[nist_algos['Scheme'] == scheme]

    if len(match) > 0:
        row = match.iloc[0]
        table_iii_data.append({
            'Scheme': row_ii['Scheme'],
            'Family': row_ii['Family'],
            'Class 0 (4KB)': f"{row['EAS_Class_0']:.3f}",
            'Class 1 (10KB)': f"{row['EAS_Class_1']:.3f}",
            'Class 2 (50KB)': f"{row['EAS_Class_2']:.3f}"
        })

table_iii_df = pd.DataFrame(table_iii_data)
display(table_iii_df)

print("\n💡 Suitability Legend:")
print("   0.000 = Incompatible (exceeds limits)")
print("   0.001-0.500 = Heavy (high strain)")
print("   0.501-1.000 = Compatible (functional)")
print("   >1.000 = Optimal (best choice)")

## 📊 Part 10: Comprehensive Table (All NIST Algorithms)

In [ ]:
print("\n" + "="*80)
print("📊 SUPPLEMENTARY TABLE: All NIST Algorithms (for appendix)")
print("="*80)

# Create comprehensive table
supp_data = []

for idx, row in nist_algos.iterrows():
    supp_data.append({
        'Scheme': row['Scheme'],
        'Family': row['NIST_Family'],
        'Implementation': row['Implementation'],
        'Security': int(row['Security_Level']),
        'Cycles': f"{int(row['Total_Cycles']):,}",
        'RAM (KB)': f"{row['Max_RAM']/1024:.1f}",
        'Flash (KB)': f"{row['Flash']/1024:.1f}",
        'EAS C0': f"{row['EAS_Class_0']:.3f}",
        'EAS C1': f"{row['EAS_Class_1']:.3f}",
        'EAS C2': f"{row['EAS_Class_2']:.3f}"
    })

supp_df = pd.DataFrame(supp_data)
display(supp_df)

print(f"\n✅ Complete NIST dataset: {len(supp_df)} algorithms")
print("   (Include this as supplementary material)")

## 📈 Part 11: Visualization - Top NIST Algorithms

In [ ]:
# Get top 10 NIST algorithms by EAS (Class 2)
top10 = nist_algos.nlargest(10, 'EAS_Class_2')

fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(top10))
width = 0.25

# Shorten labels if needed
labels = [s[:30] + '...' if len(s) > 30 else s for s in top10['Scheme']]

ax.bar(x - width, top10['EAS_Class_0'], width, label='Class 0 (4KB)', color='#e74c3c', alpha=0.8)
ax.bar(x, top10['EAS_Class_1'], width, label='Class 1 (10KB)', color='#f39c12', alpha=0.8)
ax.bar(x + width, top10['EAS_Class_2'], width, label='Class 2 (50KB)', color='#27ae60', alpha=0.8)

ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_ylabel('EAS-PQC Score', fontsize=12, fontweight='bold')
ax.set_title('Top 10 NIST PQC Algorithms by EAS Score', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Heavy/Compatible')
ax.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Compatible/Optimal')

plt.tight_layout()
plt.show()

print("✅ Visualization complete!")

## 📈 Part 12: Visualization - By NIST Family

In [ ]:
# Average EAS by NIST family
family_avg = nist_algos.groupby('NIST_Family')['EAS_Class_2'].mean().sort_values(ascending=False)
family_count = nist_algos.groupby('NIST_Family').size()

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(family_avg)))
bars = ax.barh(family_avg.index, family_avg.values, color=colors, edgecolor='black', alpha=0.8)

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    family_name = family_avg.index[i]
    count = family_count[family_name]
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2.,
            f'{width:.3f} (n={count})',
            ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Average EAS Score (Class 2)', fontsize=12, fontweight='bold')
ax.set_title('NIST Algorithm Family Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Family comparison complete!")

## 💾 Part 13: Save Results

In [ ]:
# Save all tables
table_ii_df.to_csv('table_ii_nist_representative.csv', index=False)
table_iii_df.to_csv('table_iii_nist_eas_scores.csv', index=False)
supp_df.to_csv('supplementary_all_nist_algorithms.csv', index=False)
nist_algos.to_csv('nist_algorithms_full_data.csv', index=False)

print("💾 Saved files:")
print("   1. table_ii_nist_representative.csv (for paper Table II)")
print("   2. table_iii_nist_eas_scores.csv (for paper Table III)")
print("   3. supplementary_all_nist_algorithms.csv (for appendix)")
print("   4. nist_algorithms_full_data.csv (complete dataset)")

# Download
from google.colab import files
files.download('table_ii_nist_representative.csv')
files.download('table_iii_nist_eas_scores.csv')
files.download('supplementary_all_nist_algorithms.csv')
files.download('nist_algorithms_full_data.csv')

print("\n✅ All files downloaded!")

In [ ]:
print("="*80)
print("🔍 OUTPUT VALIDATION - Verify Against Paper Tables")
print("="*80)

# Ground truth from paper Table II (corrected values)
PAPER_TABLE_II = {
    'ml-kem-512': {'cycles': 2185051, 'ram': 9560, 'flash': 5116},
    'ml-kem-1024': {'cycles': 5264741, 'ram': 20352, 'flash': 6160},
    'sphincs-sha2-128f-simple': {'cycles': 406241846, 'ram': 4956, 'flash': 4956},
    'hqc-128': {'cycles': 317925277, 'ram': 55892, 'flash': 18628},
    'ml-dsa-44': {'cycles': 11863456, 'ram': 51976, 'flash': 8212},
    'bikel1': {'cycles': 87489677, 'ram': 181161, 'flash': 181161},
}

print("\nTable II Validation:")
print(f"{'Scheme':<40} {'Cycles':<12} {'RAM':<10} {'Flash':<10}")
print("-"*75)

errors = []
for scheme, expected in PAPER_TABLE_II.items():
    row = nist_algos[nist_algos['Scheme'] == scheme]
    if len(row) > 0:
        row = row.iloc[0]
        cyc_match = int(row['Total_Cycles']) == expected['cycles']
        ram_match = int(row['Max_RAM']) == expected['ram']
        flash_match = int(row['Flash']) == expected['flash']

        cyc_ok = '✅' if cyc_match else f"❌ {int(row['Total_Cycles'])}"
        ram_ok = '✅' if ram_match else f"❌ {int(row['Max_RAM'])}"
        flash_ok = '✅' if flash_match else f"❌ {int(row['Flash'])}"

        print(f"{scheme:<40} {cyc_ok:<12} {ram_ok:<10} {flash_ok:<10}")

        if not (cyc_match and ram_match and flash_match):
            errors.append(scheme)

if not errors:
    print("\n" + "="*75)
    print("✅ ✅ ✅  ALL VALUES MATCH PAPER TABLE II  ✅ ✅ ✅")
    print("="*75)
    print("\n🎉 Code is fully corrected and reproducible!")
else:
    print(f"\n❌ ERRORS FOUND IN: {', '.join(errors)}")
    print("   Check that all bug fixes were applied correctly.")


## 🎯 Part 14: Summary & Paper Recommendations

In [ ]:
print("="*80)
print("🎯 FINAL SUMMARY")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"   Total algorithms in CSV: {df_all['Scheme'].nunique()}")
print(f"   NIST algorithms found: {nist_algos['Scheme'].nunique()}")
print(f"   Non-NIST excluded: {df_all['Scheme'].nunique() - nist_algos['Scheme'].nunique()}")

print(f"\n📋 NIST Algorithm Breakdown:")
for family in nist_algos['NIST_Family'].unique():
    count = len(nist_algos[nist_algos['NIST_Family'] == family])
    print(f"   {family:<45} → {count} algorithms")

print(f"\n📄 For Your Paper:")
print(f"   Table II: {len(table_ii_df)} representative NIST algorithms")
print(f"   Table III: EAS scores for same {len(table_iii_df)} algorithms")
print(f"   Supplementary: All {len(supp_df)} NIST algorithms")

print("\n📝 Add to Paper Methodology Section:")
print("-"*80)
print("""
\\subsection{Algorithm Selection}

We evaluate NIST-standardized Post-Quantum Cryptography algorithms
(FIPS 203-205) and selected Round 4 finalists. This selection represents
the algorithms most likely to be deployed in real-world IoT systems,
ensuring our findings are directly applicable to industry implementations.

Our evaluation includes:
\\begin{itemize}
    \\item ML-KEM variants (FIPS 203): All parameter sets
    \\item ML-DSA variants (FIPS 204): All parameter sets
    \\item SLH-DSA variants (FIPS 205): Representative parameter sets
    \\item BIKE: Round 4 code-based KEM candidate
    \\item HQC: Round 4 code-based KEM candidate
    \\item Falcon (FN-DSA): Round 4 lattice signature candidate
\\end{itemize}

While the PQM4 benchmark suite contains 92 cryptographic schemes, we
focus on the 23 NIST-designated algorithms to maintain relevance to
standards-compliant IoT deployments.
""")

print("="*80)
print("✅ Analysis Complete!")
print("\n🎓 Your paper now focuses on NIST algorithms only - industry relevant!")